<a href="https://colab.research.google.com/github/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/blob/main/my%20training%20code/Step2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import yaml
import re
import time
import urllib.request
from sentence_transformers import SentenceTransformer
import torch
import umap
import plotly.express as px
from sklearn.cluster import KMeans
import numpy as np

print("🚀 啟動階段二：動態地名脫敏與高維語義探索")

# ==========================================
# 1. 載入 YAML 設定 (直接從 GitHub)
# ==========================================
def load_sanitization_config(yaml_url):
    response = urllib.request.urlopen(yaml_url)
    config = yaml.safe_load(response)

    loc_config = config.get('locations', {})
    all_locations = []
    for category in loc_config:
        all_locations.extend(loc_config[category])

    if not all_locations:
        return None
    pattern_str = f"({'|'.join(all_locations)})"
    return re.compile(pattern_str)

YAML_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml"
city_pattern = load_sanitization_config(YAML_URL)
if city_pattern:
    print(f"✅ 已從 YAML 載入 {city_pattern.groups} 個地名特徵進行防禦。")
else:
    print("⚠️ 未找到地名設定。")

# ==========================================
# 2. 脫敏處理函數
# ==========================================
def preprocess_texts(texts, replacement="[LOC]"):
    if not city_pattern: return texts
    if isinstance(texts, str):
        return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

# ==========================================
# 3. 執行資料清洗 (直接從 GitHub 讀取候選池)
# ==========================================
print("\n載入候選池資料並執行脫敏...")
# ⚠️ 請確保你在 Step 1 產生的檔案已經上傳到 GitHub，並把網址貼在這裡
# 這裡先預設你的檔案名稱為 candidate_pool_sanitized.csv
CANDIDATE_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/candidate_pool_sanitized.csv"

df = pd.read_csv(CANDIDATE_URL)
df = df.dropna(subset=['name']).drop_duplicates(subset=['name']).reset_index(drop=True)

start_time = time.time()
df['original_name'] = df['name']
df['name'] = preprocess_texts(df['name'].tolist())
print(f"✅ 脫敏完成，耗時: {time.time() - start_time:.2f} 秒")

# ==========================================
# 4. 生成 Embeddings
# ==========================================
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n使用的運算裝置: {device}")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

texts = df['name'].tolist()
print("🚀 開始生成 Embeddings... ")
start_time = time.time()
embeddings = model.encode(texts, batch_size=512, show_progress_bar=True)
print(f"✅ 向量化完成！耗時: {time.time() - start_time:.2f} 秒")

# ==========================================
# 5. UMAP 降維與視覺化 (優化參數)
# ==========================================
print("\n正在進行 UMAP 降維... (約需 10~30 秒)")
# 🌟 優化點：n_neighbors 調大為 30，讓群集更緊密
reducer = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1, random_state=42, metric='cosine')
embeddings_2d = reducer.fit_transform(embeddings)

df['x'] = embeddings_2d[:, 0]
df['y'] = embeddings_2d[:, 1]
df['z'] = embeddings_2d[:, 2]

# 🌟 優化點：直接將類別轉為對應的英文標籤，為 Step 3 鋪路
def get_color_category(row):
    if row.get('matched_loan', False): return 'loan'
    if row.get('matched_porn', False): return 'porn'
    if row.get('matched_gambling', False): return 'gambling'
    if row.get('matched_scam_recruitment', False): return 'scam_recruitment'
    return 'multiple'

df['color_category'] = df.apply(get_color_category, axis=1)

# ==========================================
# 6. K-Means 採樣與儲存
# ==========================================
print("\n啟動黃金樣本自動採樣 (K-Means Centroid Sampling)...")
gold_candidates_list = []
categories = df['color_category'].unique()

for cat in categories:
    if cat == 'multiple': continue

    print(f"處理類別: {cat}")
    cat_df = df[df['color_category'] == cat].copy()
    cat_indices = cat_df.index.tolist()
    cat_embeddings = embeddings[cat_indices]

    # 動態決定群數 (最多 30 群)
    n_clusters = min(30, max(1, len(cat_df) // 10))

    if n_clusters == 0: continue

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(cat_embeddings)
    distances = kmeans.transform(cat_embeddings)

    for i in range(n_clusters):
        cluster_mask = kmeans.labels_ == i
        cluster_distances = distances[cluster_mask, i]
        top_k = min(5, sum(cluster_mask))
        closest_local_indices = np.argsort(cluster_distances)[:top_k]
        actual_indices = np.where(cluster_mask)[0][closest_local_indices]

        for idx in actual_indices:
            row = cat_df.iloc[idx]
            gold_candidates_list.append({
                'text': row['name'],
                'proposed_category': cat, # 這裡現在存的是英文 ('loan', 'porn' 等)
                'cluster_id': i
            })

df_gold_candidates = pd.DataFrame(gold_candidates_list)
print(f"\n✅ 採樣完成！共萃取出 {len(df_gold_candidates)} 筆高品質候選樣本。")
df_gold_candidates.to_csv('gold_candidates_review.csv', index=False, encoding='utf-8-sig')

print("\n💡 下一步：請下載 gold_candidates_review.csv 進行人工審核，並將審核完畢的檔案上傳回 GitHub。")

🚀 啟動階段二：動態地名脫敏與高維語義探索
✅ 已從 YAML 載入 1 個地名特徵進行防禦。

載入候選池資料並執行脫敏...
✅ 脫敏完成，耗時: 0.01 秒

使用的運算裝置: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 開始生成 Embeddings... 


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ 向量化完成！耗時: 101.61 秒

正在進行 UMAP 降維... (約需 10~30 秒)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.




啟動黃金樣本自動採樣 (K-Means Centroid Sampling)...
處理類別: porn
處理類別: scam_recruitment
處理類別: loan
處理類別: gambling

✅ 採樣完成！共萃取出 507 筆高品質候選樣本。

💡 下一步：請下載 gold_candidates_review.csv 進行人工審核，並將審核完畢的檔案上傳回 GitHub。


In [4]:
import plotly.express as px

# 確保之前已經跑完了 UMAP 降維的這一段：
# reducer = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1, random_state=42, metric='cosine')
# embeddings_2d = reducer.fit_transform(embeddings)
# df['x'] = embeddings_2d[:, 0]
# df['y'] = embeddings_2d[:, 1]
# df['z'] = embeddings_2d[:, 2]
# df['color_category'] = ...

print("\n🎨 開始繪製 3D 意圖特徵空間流形分佈圖...")

# 建立一個中英對照的 dictionary 讓圖表的圖例更美觀
label_translation = {
    'loan': '借貸融資',
    'porn': '黃色與特種行業',
    'gambling': '博弈',
    'scam_recruitment': '詐騙高風險招募',
    'multiple': '多重觸發 (邊界資料)'
}

# 為了繪圖好看，我們新增一個中文的類別欄位
df['display_category'] = df['color_category'].map(label_translation)

# 繪製 Plotly 互動式 3D 散佈圖
fig = px.scatter_3d(
    df,
    x='x', y='y', z='z',
    color='display_category',  # 使用中文類別作為顏色區分
    hover_data=['name'],       # 滑鼠移過去時顯示原始社團名稱
    title='意圖特徵空間流形分佈 (3D UMAP Projection)',
    opacity=0.7,
    # 使用一套清晰的調色盤
    color_discrete_sequence=px.colors.qualitative.Plotly,
    width=1000, height=800
)

# 調整點的大小，讓圖看起來不會太密集
fig.update_traces(marker=dict(size=3))

# 隱藏座標軸數字 (因為 UMAP 的絕對座標沒有意義，群集的相對距離才有意義)
fig.update_xaxes(showticklabels=False, title='')
fig.update_yaxes(showticklabels=False, title='')

# 設定背景為白色，比較乾淨
fig.update_layout(
    width=1000,
    height=800,
    template="plotly_white",
    legend_title_text='預測類別'
)

# 在 Colab 顯示圖表
fig.show()


🎨 開始繪製 3D 意圖特徵空間流形分佈圖...
